# Task 3: Cross-Encoder JobBERT (Colab GPU)

**Runtime → Change runtime type → T4 GPU.** Notebook ini hanya menjalankan `task3_match/cross_encoder.py`; semua logika ada di skrip itu.

Alur (sesuai TASKS.md, Task 3):
1. Tuning lr ∈ {2e-5, 3e-5} hanya pada **V1** (raw text), untuk run `cls` (3 kelas, CE) dan `reg` (skor/100, MSE). Pemilihan pakai **val** (Macro-F1 untuk cls, MAE untuk reg), bukan test.
2. **V2** (+ entity Task 1) dan **V3** (+ profil Task 2) dengan lr terpilih → ablation memakai split, hyperparameter, dan seed yang sama.
3. Tiap run: maks 4 epoch, early stopping (patience 1), batch 16, warmup 10%, weight decay 0,01, fp16.

Hasil (CSV + prediksi + model) disalin ke Google Drive setelah tiap run; run yang sudah ada di Drive dilewati (aman jika Colab terputus).
V3 saat ini memakai **profil stub** (`data/cache/profiles_stub.parquet`) → hasil bertanda `(stub)`. Ganti `PROFILES` setelah Task 2 menyerahkan profil 778 resume.

In [ ]:
BRANCH = "feat/task3-match"          # branch yang berisi task3_match/
COPY_MODELS = True                   # ~420 MB per run di Drive; False = hanya CSV & prediksi
PROFILES = "profiles_stub.parquet"   # file di data/cache/

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/task3_match"
!mkdir -p {OUT}/results {OUT}/models
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b {BRANCH} https://github.com/Ferdin-Arsenic/Job_Matching_System-NLP-.git repo
%cd /content/repo
!pip install -q datasets accelerate
!ls data/splits data/cache outputs/ner

In [ ]:
import os, glob, subprocess, pandas as pd

def run(variant, kind, lr):
    tag = f"{variant}{'-stub' if variant == 'V3' and 'stub' in PROFILES else ''}_{kind}_lr{lr:g}"
    if os.path.exists(f"{OUT}/results/task3_ce_{tag}.csv"):
        print("skip (sudah ada di Drive):", tag); return
    subprocess.run(["python", "task3_match/cross_encoder.py", "--variant", variant, "--run", kind, "--lr", str(lr),
                    "--profiles", PROFILES], check=True)
    os.system(f"cp data/results/task3_ce_{tag}.csv data/results/task3_pred_{tag}.parquet {OUT}/results/")
    if COPY_MODELS: os.system(f"cp -r models/match/{tag} {OUT}/models/")

def results():
    return pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/results/task3_ce_*.csv")], ignore_index=True)

## 1. Tuning lr di V1 (4 run)

In [ ]:
for kind in ["cls", "reg"]:
    for lr in [2e-5, 3e-5]:
        run("V1", kind, lr)

In [ ]:
r = results().query("variant == 'V1' and split == 'val'")
best = {"cls": r.query("run == 'cls' and metric == 'macro_F1'").sort_values("value", ascending=False).lr.iloc[0],
        "reg": r.query("run == 'reg' and metric == 'MAE'").sort_values("value").lr.iloc[0]}
print("lr terpilih (val):", best)
r.query("metric in ['macro_F1', 'MAE', 'ROC_AUC', 'NDCG@5']").pivot_table(index=["run", "lr"], columns="metric", values="value").round(3)

## 2. Ablation V2, V3 dengan lr terpilih (4 run)

In [ ]:
for v in ["V2", "V3"]:
    for kind in ["cls", "reg"]:
        run(v, kind, best[kind])

## 3. Ringkasan test (V1 ditampilkan dengan lr terpilih)

In [ ]:
r = results().query("split == 'test'")
r = r[r.apply(lambda x: x.lr == best[x.run], axis=1)]
cols = ["macro_F1", "ROC_AUC", "MAE", "Spearman", "Spearman_per_job", "NDCG@5", "MRR", "macro_F1_threshold"]
r.pivot_table(index=["run", "model"], columns="metric", values="value")[cols].round(3)

Setelah selesai: unduh folder `MyDrive/task3_match/results/` ke `data/results/` di repo lokal (dan `models/` ke `models/match/` jika dipakai CLI).